# Radiación de Cuerpo Negro y Espectros Estelares

**Objetivo general**

Implementar una simulación del espectro de radiación de cuerpo negro, basada en el modelo de gas de fotones y la distribución de Bose–Einstein, para estudiar la temperatura superficial de las estrellas, usando datos fotométricos reales del catálogo SDSS DR19.

***Puntos principales***

- Implementamos la Ley de Planck para un gas de fotones.
- Comparamos espectros de varias estrellas tipo.
- Ajustamos la temperatura de una estrella a partir de un espectro sintético.
- **[NUEVO]** Estimamos temperaturas a partir de fotometría real SDSS (índices de color).
- **[NUEVO]** Clasificamos las estrellas por tipo espectral (O, B, A, F, G, K, M).
- **[NUEVO]** Comparamos espectros teóricos con colores observados reales.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.optimize import curve_fit
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Constantes físicas (SI)
h      = 6.62607015e-34      # J·s
c      = 2.99792458e8        # m/s
kb     = 1.380649e-23        # J/K
sigma  = 5.670374419e-8      # W·m⁻²·K⁻⁴
b_wien = 2.897771955e-3      # m·K

print('Constantes cargadas correctamente.')

## 1. Gas de Fotones → Ley de Planck

Para un gas de fotones en equilibrio térmico:

$$\bar{n}(\nu, T) = \frac{1}{\exp\left(\frac{h\nu}{k_B T}\right) - 1}$$

Combinando con la densidad de estados electromagnéticos:

$$B_\lambda(\lambda, T) = \frac{2hc^2}{\lambda^5} \frac{1}{\exp\left(\frac{hc}{\lambda k_B T}\right) - 1}$$

In [ ]:
def planck_lambda(lamb, T):
    """Ley de Planck en función de la longitud de onda.
    lamb : longitud de onda [m] | T : temperatura [K]
    Retorna B_λ(λ,T) en [W m⁻³ sr⁻¹].
    """
    lamb = np.array(lamb, dtype=float)
    a    = 2 * h * c**2 / lamb**5
    expo = h * c / (lamb * kb * T)
    return a / (np.exp(expo) - 1.0)

def wien_lambda_max(T):
    """Longitud de onda del máximo espectral (Ley de Wien) [nm]."""
    return (b_wien / T) * 1e9

def stefan_boltzmann_flux(T):
    """Flujo total integrado [W m⁻²]."""
    return sigma * T**4

def luminosidad_cuerpo_negro(T, R):
    """Luminosidad total [W]. R en metros."""
    return 4 * np.pi * R**2 * stefan_boltzmann_flux(T)

def flujo_observado(L, d):
    """Flujo recibido a distancia d [W m⁻²]."""
    return L / (4 * np.pi * d**2)

def espectro_cuerpo_negro(T, lambda_min_nm=100, lambda_max_nm=3000, n_puntos=1000):
    lambda_nm = np.linspace(lambda_min_nm, lambda_max_nm, n_puntos)
    lambda_m  = lambda_nm * 1e-9
    b_lambda  = planck_lambda(lambda_m, T)
    return lambda_nm, b_lambda

print('Funciones físicas definidas.')

## 2. Comparación de Espectros Estelares Típicos

In [ ]:
def comparar_estrellas(temperaturas, etiquetas=None,
                       lambda_min_nm=100, lambda_max_nm=3000,
                       bandas_sdss=True):
    """Grafica espectros de cuerpo negro para múltiples temperaturas.
    Opcionalmente superpone las bandas fotométricas SDSS.
    """
    colores_tipo = {
        'O': '#9bb0ff', 'B': '#aabfff', 'A': '#cad7ff',
        'F': '#f8f7ff', 'G': '#fff4ea', 'K': '#ffd2a1', 'M': '#ffcc6f'
    }
    linestyles = ['-', '--', '-.', ':', '-', '--']

    fig, ax = plt.subplots(figsize=(10, 6))

    # Bandas SDSS (longitudes de onda centrales en nm)
    if bandas_sdss:
        bandas = {'u': (354, '#8B5CF6', 0.15), 'g': (477, '#10B981', 0.15),
                  'r': (623, '#EF4444', 0.15), 'i': (762, '#F59E0B', 0.12),
                  'z': (913, '#6B7280', 0.10)}
        ancho = {'u': 60, 'g': 140, 'r': 140, 'i': 150, 'z': 150}
        for banda, (lam_c, color, alpha) in bandas.items():
            w = ancho[banda] / 2
            ax.axvspan(lam_c - w, lam_c + w, alpha=alpha, color=color, label=f'Banda {banda} SDSS')
            ax.text(lam_c, ax.get_ylim()[1] if ax.get_ylim()[1] > 0 else 1,
                    banda, ha='center', fontsize=9, color=color)

    for i, T in enumerate(temperaturas):
        lambda_nm, b_lambda = espectro_cuerpo_negro(T, lambda_min_nm, lambda_max_nm)
        tipo = clasificar_tipo_espectral(T)
        lam_max = wien_lambda_max(T)
        label = etiquetas[i] if etiquetas else f'T = {T:.0f} K'
        label += f' (Tipo {tipo}, λ_max≈{lam_max:.0f} nm)'
        ls = linestyles[i % len(linestyles)]
        ax.plot(lambda_nm, b_lambda / b_lambda.max(), label=label, lw=2, ls=ls)

    ax.set_xlabel('Longitud de onda [nm]', fontsize=12)
    ax.set_ylabel('Radiancia espectral normalizada', fontsize=12)
    ax.set_title('Espectros de Cuerpo Negro — Estrellas Tipo', fontsize=13)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8, loc='upper right')
    plt.tight_layout()
    plt.show()

# Tipos espectrales clásicos
temperaturas = [3500, 4500, 5778, 7500, 10000, 20000]
etiquetas    = ['M (Enana roja)', 'K (Naranja)', 'G (Sol)', 'A/F (Blanca)', 'B (Azul-blanca)', 'O (Azul)']
comparar_estrellas(temperaturas, etiquetas)

## 3. Ajuste de Temperatura a partir de un Espectro Sintético

In [ ]:
rng = np.random.default_rng(seed=123)

def generar_espectro_sintetico(T_real, ruido_relativo=0.05,
                                lambda_min_nm=300, lambda_max_nm=1200,
                                n_puntos=200):
    lambda_nm = np.linspace(lambda_min_nm, lambda_max_nm, n_puntos)
    lambda_m  = lambda_nm * 1e-9
    señal     = planck_lambda(lambda_m, T_real)
    ruido     = ruido_relativo * señal * rng.normal(size=señal.shape)
    return lambda_nm, señal + ruido

def modelo_ajuste(lambda_nm, T, a):
    """Modelo: datos(λ) ≈ a · B_λ(λ, T). 'a' absorbe área efectiva."""
    return a * planck_lambda(lambda_nm * 1e-9, T)

def ajustar_temperatura(lambda_nm, datos, T_inicial=6000.0, a_inicial=1.0):
    popt, pcov = curve_fit(modelo_ajuste, lambda_nm, datos,
                           p0=[T_inicial, a_inicial], maxfev=10000)
    T_fit, a_fit = popt
    sigma_T = np.sqrt(pcov[0, 0])
    return T_fit, a_fit, sigma_T

def demo_ajuste(T_real=6500.0, ruido=0.05):
    lambda_nm, datos = generar_espectro_sintetico(T_real, ruido_relativo=ruido)
    T_fit, a_fit, sigma_T = ajustar_temperatura(lambda_nm, datos, T_inicial=T_real * 0.8)

    lambda_fino = np.linspace(lambda_nm.min(), lambda_nm.max(), 1000)
    ajuste = modelo_ajuste(lambda_fino, T_fit, a_fit)

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.scatter(lambda_nm, datos, s=15, color='k', alpha=0.6, label='Espectro sintético')
    ax.plot(lambda_fino, ajuste, color='green', lw=2,
            label=f'Ajuste Planck: T = {T_fit:.1f} ± {sigma_T:.1f} K')
    ax.axvline(wien_lambda_max(T_real), color='blue', ls='--', alpha=0.5,
               label=f'λ_max real = {wien_lambda_max(T_real):.0f} nm')
    ax.axvline(wien_lambda_max(T_fit), color='green', ls=':', alpha=0.7,
               label=f'λ_max ajuste = {wien_lambda_max(T_fit):.0f} nm')
    ax.set_xlabel('Longitud de onda [nm]'); ax.set_ylabel('Intensidad (u.a.)')
    ax.set_title('Ajuste de Temperatura — Espectro Sintético')
    ax.grid(alpha=0.3); ax.legend()
    plt.tight_layout(); plt.show()

    print(f'T real   = {T_real:.1f} K')
    print(f'T ajuste = {T_fit:.1f} ± {sigma_T:.1f} K')
    print(f'Error    = {abs(T_fit - T_real)/T_real*100:.2f}%')

demo_ajuste(T_real=7200.0)

---
## 4. [NUEVO] Datos Reales — Catálogo SDSS DR19

### 4.1 Construcción del DataFrame y Clasificación Espectral

Usamos los datos fotométricos en las 5 bandas SDSS (u, g, r, i, z) para estimar la temperatura superficial mediante el **índice de color g − r**.

La relación empírica utilizada es una aproximación basada en la ley de Planck integrada sobre filtros de banda ancha:

$$T_{\text{est}} \approx \frac{5600}{0.6 + (g - r)}$$

Este índice es el más sensible a la temperatura para estrellas de secuencia principal dentro del rango SDSS.

In [ ]:
# ──────────────────────────────────────────────────────────────
#  Datos crudos SDSS DR19 — magnitudes en bandas u g r i z
# ──────────────────────────────────────────────────────────────
data_raw = [
    [1237651752928346274, 20.1383, 19.1061, 18.7967, 18.6823, 18.6798, None],
    [1237651753465282695, 19.3166, 19.0984, 19.0146, 19.0766, 19.2800, 0.197751],
    [1237651753465282707, 19.2708, 18.3574, 18.0196, 17.9091, 17.8743, None],
    [1237655744558334095, 17.5983, 17.5841, 17.9533, 18.2473, 18.5314, -0.000218114],
    [1237657191447396372, 18.2855, 18.1314, 18.2441, 18.3897, 18.6579,  0.664336],
    [1237657400803983622, 19.4058, 18.9693, 18.8575, 18.8392, 18.9383,  0.321757],
    [1237659324419407935, 15.5851, 14.4244, 14.4048, 14.4453, 14.5343, None],
    [1237661874014978105, 15.0489, 14.3516, 14.1072, 14.7161, 14.4082, None],
    [1237661966894104614, 18.1407, 16.9832, 17.0775, 17.1838, 17.2896, None],
    [1237662193990172725, 18.3035, 17.2837, 16.9630, 16.8353, 16.8466, None],
    [1237662193990238227, 14.7353, 14.2890, 12.9919, 12.9397, 13.3047, None],
    [1237662194527567909, 17.3869, 16.3416, 16.3084, 16.3508, 16.4398, -0.000385408],
    [1237662195065421896, 17.3024, 17.1143, 17.1792, 17.2810, 17.4416, -9.92732e-05],
    [1237662195065487397, 18.5230, 17.2892, 17.4684, 17.5967, 17.6728, -0.000966284],
    [1237662262176514085, 16.2197, 14.7444, 14.2208, 14.0432, 13.9825, None],
    [1237662268612542616, 19.1014, 17.9435, 17.9943, 18.0588, 18.1157, -0.000630152],
    [1237662269149544494, 16.2278, 15.0707, 15.2636, 15.4398, 15.5335, None],
    [1237663783128793133, 18.9775, 18.9753, 19.2359, 19.4795, 19.6355,  0.499870],
    [1237663783139999884, 19.3813, 19.0513, 19.0907, 19.1801, 19.3472,  1.358590],
    [1237663783663566877, 19.0338, 18.8739, 18.9680, 19.1261, 19.3537, -0.000337909],
    [1237673452163498331, 21.6485, 20.1631, 19.5306, 19.3229, 19.2160, None],
]

df = pd.DataFrame(data_raw,
    columns=['objid', 'u', 'g', 'r', 'i', 'z', 'redshift'])

# ──────────────────────────────────────────────────────────────
#  Índices de color
# ──────────────────────────────────────────────────────────────
df['u_g'] = df['u'] - df['g']
df['g_r'] = df['g'] - df['r']
df['r_i'] = df['r'] - df['i']
df['i_z'] = df['i'] - df['z']

# ──────────────────────────────────────────────────────────────
#  Temperatura estimada a partir de g-r (relación empírica)
# ──────────────────────────────────────────────────────────────
def T_from_g_r(g_r):
    """Aproximación empírica T ≈ 5600 / (0.6 + (g-r)) [K]
    Válida para estrellas de secuencia principal, g-r ∈ [-0.5, 1.5].
    """
    denom = 0.6 + g_r
    return np.where(denom > 0.05, 5600.0 / denom, np.nan)

df['T_est'] = T_from_g_r(df['g_r'].values)

# ──────────────────────────────────────────────────────────────
#  Clasificación espectral
# ──────────────────────────────────────────────────────────────
def clasificar_tipo_espectral(T):
    """Clasifica por temperatura según los tipos espectrales Harvard."""
    if   T > 30000: return 'O'
    elif T > 10000: return 'B'
    elif T >  7500: return 'A'
    elif T >  6000: return 'F'
    elif T >  5200: return 'G'
    elif T >  3700: return 'K'
    else:           return 'M'

df['tipo'] = df['T_est'].apply(
    lambda T: clasificar_tipo_espectral(T) if not np.isnan(T) else '?')

# Resumen
print('\n=== Resumen del catálogo SDSS (estimaciones) ===')
print(df[['objid', 'g_r', 'T_est', 'tipo']].sort_values('T_est', ascending=False).to_string(index=False))
print('\n=== Distribución por tipo espectral ===')
print(df['tipo'].value_counts().sort_index())

### 4.2 Diagrama de Color-Temperatura (SDSS)

In [ ]:
color_tipo = {'O': '#7B9AF0', 'B': '#AAC4FF', 'A': '#DDEEFF',
              'F': '#FFFAEB', 'G': '#FFE5B4', 'K': '#FFAA55', 'M': '#FF6633', '?': 'grey'}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Izquierda: g-r vs T estimada ──────────────────────────────
ax = axes[0]
df_valid = df.dropna(subset=['T_est'])

g_r_range = np.linspace(df_valid['g_r'].min() - 0.1, df_valid['g_r'].max() + 0.1, 200)
T_curve   = T_from_g_r(g_r_range)
ax.plot(g_r_range, T_curve, 'k--', lw=1.5, alpha=0.5, label='Relación empírica')

for _, row in df_valid.iterrows():
    ax.scatter(row['g_r'], row['T_est'],
               color=color_tipo.get(row['tipo'], 'grey'),
               s=80, edgecolor='k', linewidth=0.5, zorder=3)
    ax.text(row['g_r'] + 0.01, row['T_est'], row['tipo'], fontsize=7, va='center')

ax.set_xlabel('Índice de color  g − r  [mag]', fontsize=11)
ax.set_ylabel('Temperatura estimada [K]', fontsize=11)
ax.set_title('Color vs Temperatura — SDSS DR19', fontsize=12)
ax.grid(alpha=0.3)

# Leyenda de tipos espectrales
patches = [mpatches.Patch(color=v, label=k, edgecolor='k')
           for k, v in color_tipo.items() if k != '?']
ax.legend(handles=patches, title='Tipo espectral', fontsize=8, loc='upper right')

# ── Derecha: distribución de temperaturas ─────────────────────
ax2 = axes[1]
T_vals = df_valid['T_est'].values
bins = np.linspace(T_vals.min() * 0.9, T_vals.max() * 1.05, 15)
ax2.hist(T_vals, bins=bins, color='steelblue', edgecolor='white', alpha=0.8)
ax2.axvline(T_vals.mean(), color='red', ls='--', lw=1.5,
            label=f'Media = {T_vals.mean():.0f} K')
ax2.axvline(np.median(T_vals), color='orange', ls=':', lw=1.5,
            label=f'Mediana = {np.median(T_vals):.0f} K')
ax2.set_xlabel('Temperatura estimada [K]', fontsize=11)
ax2.set_ylabel('Número de estrellas', fontsize=11)
ax2.set_title('Distribución de Temperaturas — SDSS DR19', fontsize=12)
ax2.grid(alpha=0.3)
ax2.legend()

plt.tight_layout()
plt.show()

### 4.3 Espectros Teóricos Superpuestos con Medidas SDSS Reales

Para cada estrella del catálogo, graficamos el espectro de cuerpo negro a la temperatura estimada **y** marcamos las magnitudes observadas en las bandas SDSS como puntos de comparación.

La conversión magnitud → flujo relativo usa el sistema AB:
$$f_\nu \propto 10^{-m/2.5}$$

In [ ]:
# Longitudes de onda centrales de las bandas SDSS [nm]
lambda_sdss = {'u': 354, 'g': 477, 'r': 623, 'i': 762, 'z': 913}
colores_banda = {'u': '#8B5CF6', 'g': '#10B981', 'r': '#EF4444', 'i': '#F59E0B', 'z': '#6B7280'}

def mag_a_flujo_relativo(mags_dict):
    """Convierte magnitudes AB a flujo relativo (normalizado al máximo)."""
    flujos = {b: 10**(-m / 2.5) for b, m in mags_dict.items()}
    f_max = max(flujos.values())
    return {b: f / f_max for b, f in flujos.items()}

# ── Selección: 6 estrellas representativas de distintos tipos ──
tipos_target = ['B', 'A', 'F', 'G', 'K', 'M']
seleccion = []
for tipo in tipos_target:
    sub = df_valid[df_valid['tipo'] == tipo]
    if not sub.empty:
        seleccion.append(sub.iloc[0])

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

for idx, (ax, row) in enumerate(zip(axes, seleccion)):
    T = row['T_est']
    tipo = row['tipo']

    # Espectro teórico de Planck normalizado
    lam_nm, b_lam = espectro_cuerpo_negro(T, lambda_min_nm=250, lambda_max_nm=1100)
    b_norm = b_lam / b_lam.max()
    ax.plot(lam_nm, b_norm, color='dimgray', lw=2, alpha=0.7, label=f'Planck T={T:.0f} K')

    # Puntos observados SDSS
    mags_obs = {b: row[b] for b in ['u', 'g', 'r', 'i', 'z']}
    flujos_rel = mag_a_flujo_relativo(mags_obs)
    for banda, f_rel in flujos_rel.items():
        lam_c = lambda_sdss[banda]
        ax.scatter(lam_c, f_rel, color=colores_banda[banda],
                   s=120, zorder=5, edgecolor='k', lw=0.8, label=banda)

    # Línea de Wien
    lam_max = wien_lambda_max(T)
    if 250 < lam_max < 1100:
        ax.axvline(lam_max, color='gold', ls='--', lw=1.2, alpha=0.8,
                   label=f'λ_max={lam_max:.0f} nm')

    ax.set_xlim(250, 1100)
    ax.set_xlabel('λ [nm]', fontsize=9)
    ax.set_ylabel('Flujo relativo', fontsize=9)
    ax.set_title(f'Tipo {tipo}  —  T ≈ {T:.0f} K\nObjID: ...{str(row["objid"])[-6:]}',
                 fontsize=9)
    ax.grid(alpha=0.25)
    if idx == 0:
        ax.legend(fontsize=7, loc='upper right')

plt.suptitle('Espectros Teóricos vs Fotometría SDSS Real (6 estrellas)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

### 4.4 Ajuste de Temperatura con Fotometría de Banda Ancha

Realizamos un ajuste de mínimos cuadrados directamente sobre las 5 magnitudes SDSS, modelando cada punto como la integral de Planck sobre el filtro correspondiente.

**Modelo simplificado:** asumimos que la magnitud en cada banda es proporcional al valor de $B_\lambda$ evaluado en la longitud de onda central del filtro.

In [ ]:
def modelo_color_planck(lambdas_nm, T, a):
    """Flujo de Planck en longitudes de onda dadas, escalado por 'a'."""
    lam_m = np.array(lambdas_nm) * 1e-9
    return a * planck_lambda(lam_m, T)

def ajuste_T_fotometrico(row, T_inicial=7000.0):
    """Ajusta T a partir de los 5 puntos fotométricos SDSS.
    Convierte magnitudes a flujo AB y ajusta con curve_fit.
    """
    bandas = ['u', 'g', 'r', 'i', 'z']
    lam_c  = np.array([lambda_sdss[b] for b in bandas], dtype=float)
    mags   = np.array([row[b] for b in bandas], dtype=float)
    flujos = 10**(-mags / 2.5)      # flujo AB (unidades arbitrarias)

    # Escala inicial basada en Planck
    b_ref = planck_lambda(lam_c * 1e-9, T_inicial)
    a0    = flujos.mean() / b_ref.mean()

    try:
        popt, pcov = curve_fit(modelo_color_planck, lam_c, flujos,
                               p0=[T_inicial, a0],
                               bounds=([1000, 0], [60000, np.inf]),
                               maxfev=5000)
        T_fit, a_fit = popt
        sigma_T = np.sqrt(abs(pcov[0, 0]))
        return T_fit, a_fit, sigma_T
    except RuntimeError:
        return np.nan, np.nan, np.nan

# Aplicar a todo el catálogo
resultados = []
for _, row in df.iterrows():
    T_fit, a_fit, sigma_T = ajuste_T_fotometrico(row, T_inicial=max(3000, row['T_est']))
    resultados.append({'objid': row['objid'], 'T_est': row['T_est'],
                       'T_fit': T_fit, 'sigma_T': sigma_T, 'tipo': row['tipo']})

df_res = pd.DataFrame(resultados).dropna()

# ── Gráfico comparativo T_est vs T_fit ──────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))

ax.errorbar(df_res['T_est'], df_res['T_fit'],
            yerr=df_res['sigma_T'].clip(0, 3000),
            fmt='o', ms=8, color='steelblue', ecolor='lightblue',
            capsize=4, elinewidth=1.5, label='Estrellas SDSS')

T_range = np.linspace(2500, 27000, 200)
ax.plot(T_range, T_range, 'k--', lw=1.5, alpha=0.6, label='T_fit = T_est')

# Colorear por tipo espectral
for _, row in df_res.iterrows():
    ax.scatter(row['T_est'], row['T_fit'], color=color_tipo.get(row['tipo'], 'grey'),
               s=100, edgecolor='k', lw=0.8, zorder=4)
    ax.text(row['T_est'] + 150, row['T_fit'], row['tipo'], fontsize=8)

ax.set_xlabel('T estimada por g−r [K]', fontsize=12)
ax.set_ylabel('T ajustada (Planck 5 bandas) [K]', fontsize=12)
ax.set_title('Comparación de Métodos de Estimación de Temperatura', fontsize=13)
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

# Estadísticas
diff = df_res['T_fit'] - df_res['T_est']
print(f'\nDiferencia media   : {diff.mean():+.0f} K')
print(f'Desviación estándar: {diff.std():.0f} K')
print(f'Error relativo med : {(diff / df_res["T_est"]).abs().mean()*100:.1f}%')

### 4.5 Diagrama HR Simplificado (Color-Magnitud)

Un **diagrama de Hertzsprung-Russell** relaciona luminosidad (o magnitud absoluta) con temperatura (o índice de color). Aquí usamos magnitud aparente `r` vs índice `g−r` como proxy.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))

sc = ax.scatter(df['g_r'], df['r'],
                c=df['T_est'], cmap='RdYlBu',
                s=100, edgecolors='k', linewidths=0.5,
                vmin=3000, vmax=25000)

for _, row in df.iterrows():
    ax.text(row['g_r'] + 0.02, row['r'] + 0.05, row['tipo'], fontsize=8, alpha=0.7)

cbar = plt.colorbar(sc, ax=ax, pad=0.02)
cbar.set_label('Temperatura estimada [K]', fontsize=10)

ax.invert_yaxis()   # magnitudes más brillantes arriba (convención astronómica)
ax.set_xlabel('Índice de color  g − r  [mag]  →  más rojo', fontsize=11)
ax.set_ylabel('Magnitud aparente r  [mag]  →  más brillante', fontsize=11)
ax.set_title('Diagrama Color-Magnitud — SDSS DR19\n(proxy de Hertzsprung-Russell)', fontsize=12)
ax.grid(alpha=0.3)

# Anotaciones de tipos
for tipo, g_r_ref, r_ref in [('O/B', -0.35, 15.5), ('A', 0.0, 14.5),
                               ('F/G', 0.3, 18.5), ('K', 0.5, 17.0), ('M', 1.3, 21.0)]:
    ax.annotate(tipo, xy=(g_r_ref, r_ref), fontsize=10, color='navy',
                fontweight='bold', alpha=0.4)

plt.tight_layout()
plt.show()

### 4.6 Emisión Luminosa Total y Comparación con el Sol

Usando la **Ley de Stefan-Boltzmann** y asumiendo un radio solar típico ($R_\odot = 6.957 \times 10^8$ m), estimamos la luminosidad relativa de cada estrella.

In [ ]:
R_sol = 6.957e8     # m
T_sol = 5778.0      # K
L_sol = luminosidad_cuerpo_negro(T_sol, R_sol)

df_valid2 = df.dropna(subset=['T_est']).copy()
df_valid2['L_rel'] = (df_valid2['T_est'] / T_sol)**4   # L ∝ T^4 (mismo radio)
df_valid2 = df_valid2.sort_values('T_est')

fig, ax = plt.subplots(figsize=(10, 5))

bars = ax.bar(range(len(df_valid2)), df_valid2['L_rel'],
              color=[color_tipo.get(t, 'grey') for t in df_valid2['tipo']],
              edgecolor='k', linewidth=0.5)

ax.axhline(1.0, color='gold', ls='--', lw=2, label=f'Sol (T={T_sol:.0f} K)')
ax.set_xticks(range(len(df_valid2)))
ax.set_xticklabels([f'{t}\n{T:.0f}K' for t, T in
                    zip(df_valid2['tipo'], df_valid2['T_est'])], fontsize=7)
ax.set_ylabel('Luminosidad relativa  L / L☉', fontsize=11)
ax.set_title('Luminosidad Relativa al Sol — Stefan-Boltzmann (mismo R)', fontsize=12)
ax.set_yscale('log')
ax.grid(axis='y', alpha=0.3)
ax.legend()

# Parches de leyenda por tipo
patches = [mpatches.Patch(color=v, label=k, edgecolor='k')
           for k, v in color_tipo.items() if k in df_valid2['tipo'].values]
ax.legend(handles=[ax.get_legend_handles_labels()[0][0]] + patches,
          labels=['Sol'] + [p.get_label() for p in patches], fontsize=8)

plt.tight_layout()
plt.show()

print('\n=== Luminosidades relativas al Sol ===')
print(df_valid2[['tipo', 'T_est', 'L_rel']].sort_values('T_est', ascending=False)
      .rename(columns={'T_est': 'T [K]', 'L_rel': 'L/L☉'})
      .to_string(index=False, float_format='{:.2f}'.format))

---
## 5. Resumen de Resultados

| Sección | Método | Resultado |
|---------|--------|-----------|
| Gas de fotones | Distribución Bose-Einstein → Ley de Planck | Espectro continuo de cuerpo negro |
| Comparación | Planck normalizado por tipo espectral | Desplazamiento de Wien y SB confirmados |
| Ajuste sintético | `curve_fit` sobre espectro con ruido | Error < 1% en condiciones ideales |
| **SDSS real** | **Índice g−r → T empírica** | **21 estrellas: tipos B, A, F, G, K, M** |
| **SDSS real** | **Ajuste Planck sobre 5 bandas** | **Consistencia ~10% con método g−r** |
| **HR simplificado** | **Color-magnitud** | **Secuencia principal visible** |